# yolo-traffic-pipeline — Colab runner

Esse notebook só orquestra o repositório `yolo-traffic-pipeline`; a lógica de verdade
mora nos módulos `.py` (não em células soltas), pra ficar reaproveitável fora do Colab
também.

Antes de rodar: Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4).

## 1. Clonar o repositório

Troque a URL pelo seu repositório real no GitHub.

In [ ]:
!git clone https://github.com/allvaret/yolo-traffic-pipeline.git
%cd yolo-traffic-pipeline
!pip install -q -r requirements.txt


## 2. Configurar a API key do Roboflow

Gere uma chave gratuita em roboflow.com/settings/api, depois rode a célula abaixo
e cole quando pedir (fica só na sessão, não é salva em lugar nenhum).

In [ ]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Cole sua Roboflow API key: ")


## 3. Preencher config/sources.yaml

Antes de rodar a etapa de dataset, confira se `config/sources.yaml` já tem os
`workspace`/`project`/`version` do Roboflow preenchidos pra
pedestrian_light, crosswalk e traffic_cone (edite direto pelo painel de arquivos
do Colab, ou via `%%writefile` numa célula, se preferir não editar no GitHub antes).

In [ ]:
!cat config/sources.yaml


## 4. Montar o dataset (download + conversão + dedup + split)

In [ ]:
!python pipeline.py --steps dataset


## 5. Treinar

In [ ]:
!python pipeline.py --steps train --epochs 60 --imgsz 640 --batch 16 --freeze 10


## 6. Avaliar por classe

In [ ]:
!python pipeline.py --steps evaluate


In [ ]:
import pandas as pd

df = pd.read_csv("reports/per_class_metrics.csv")
df


## 7. Salvar pesos e relatório no Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs("/content/drive/MyDrive/yolo_traffic_pipeline_outputs", exist_ok=True)
shutil.copy("runs/detect/traffic_train/weights/best.pt",
            "/content/drive/MyDrive/yolo_traffic_pipeline_outputs/best.pt")
shutil.copy("reports/per_class_metrics.csv",
            "/content/drive/MyDrive/yolo_traffic_pipeline_outputs/per_class_metrics.csv")
print("Salvo no Drive.")
